# 03 — Experimentos preliminares: Isolation Forest e indicadores físicos



In [1]:
from pathlib import Path
import os, sys, zipfile, subprocess
BASE = Path.cwd()
if not (BASE / 'iot_monitor').exists():
    git_repo = "https://github.com/jorgek14/railpulse-metropt3.git"
    if git_repo.strip():
        subprocess.run(['git', 'clone', git_repo, '/content/railpulse-metropt3'], check=True)
        BASE = Path('/content/railpulse-metropt3')
    else:
        try:
            from google.colab import files
        except ImportError as e:
            raise RuntimeError('Abra o notebook na raiz do repositório ou no Colab e envie o ZIP do projeto.') from e
        print('Selecione o ZIP do repositório RailPulse. NÃO selecione o CSV original agora.')
        enviados = files.upload()
        nome = next((n for n in enviados if n.endswith('.zip')), None)
        assert nome, 'É necessário carregar o ZIP do projeto.'
        BASE = Path('/content/railpulse-metropt3')
        BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nome) as z:
            z.extractall(BASE)
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print('Raiz do projeto:', BASE)

Raiz do projeto: /content/railpulse-metropt3


In [2]:

from google.colab import drive
from pathlib import Path
import zipfile
import shutil

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Google Drive já conectado!")
ZIP = Path('/content/drive/MyDrive/metropt+3+dataset.zip')

assert ZIP.exists(), f"Arquivo não encontrado: {ZIP}"

pasta_dados = BASE / 'dados'
pasta_dados.mkdir(parents=True, exist_ok=True)

CSV = pasta_dados / 'MetroPT3(AirCompressor).csv'

if not CSV.exists():
    with zipfile.ZipFile(ZIP, 'r') as arquivo:
        arquivos_csv = [
            nome for nome in arquivo.namelist()
            if nome.lower().endswith('.csv')
        ]

        assert arquivos_csv, "Nenhum CSV encontrado no ZIP."

        with arquivo.open(arquivos_csv[0]) as origem:
            with open(CSV, 'wb') as destino:
                shutil.copyfileobj(origem, destino)
print("✅ Base de dados carregada!")
print("Caminho:", CSV)
print("Tamanho:", round(CSV.stat().st_size / 1e6, 2), "MB")


Mounted at /content/drive
✅ Base de dados carregada!
Caminho: /content/railpulse-metropt3/dados/MetroPT3(AirCompressor).csv
Tamanho: 218.3 MB


In [3]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scikit-learn','matplotlib'],check=True)
E1=BASE/'resultados'/'etapa1'; E2=BASE/'resultados'/'etapa2'; E3=BASE/'resultados'/'etapa3'
for d in (E1,E2,E3):d.mkdir(parents=True,exist_ok=True)

## 1. Isolation Forest inicial: fevereiro para treinamento

In [4]:
subprocess.run([sys.executable,'analises/modelo_exploratorio_etapa1.py','--csv',str(CSV),'--out',str(E1)],check=True)

CompletedProcess(args=['/usr/bin/python3', 'analises/modelo_exploratorio_etapa1.py', '--csv', '/content/railpulse-metropt3/dados/MetroPT3(AirCompressor).csv', '--out', '/content/railpulse-metropt3/resultados/etapa1'], returncode=0)

## 2. Comparação de modelos globais e por regime

In [5]:
subprocess.run([sys.executable,'analises/etapa2_modelos_regime.py','--windows',str(E1/'janelas_10min_baseline_exploratorio.csv'),'--out',str(E2)],check=True)
import pandas as pd
from IPython.display import display
display(pd.read_csv(E2/'comparacao_modelos.csv'))

,modelo,janelas_teste,anomalias_janelas,taxa_anomalias_pct,episodios_total,episodios_fora_eventos,dias_observados,dias_fora_eventos,alertas_dia_observado,cal_anomalias_pct
0,global_fev,11967,2966,24.784825,53,46,102,99,0.464646,1.033592
1,global,11967,34,0.284115,1,0,102,99,0.000000,1.038753
2,regime,11967,285,2.381549,22,22,102,99,0.222222,1.078706


## 3. Engenharia de indicadores físicos e temporais

In [6]:
subprocess.run([sys.executable,'analises/etapa3_indicadores_fisicos.py','--csv',str(CSV),'--out',str(E3)],check=True)
display(pd.read_csv(E3/'comparacao_modelos.csv'))

,modelo,janelas_teste,janelas_elegiveis,cobertura_pct,janelas_sinalizadas,pct_janelas_sinalizadas,pct_elegiveis_sinalizadas,episodios_teste,episodios_fora_janelas_de_evento,dias_teste_sem_janelas_de_evento,episodios_fora_por_dia,eventos_com_novo_alerta_pre_2h
0,Fisico_IsolationForest,11967,10196,85.201,53,0.443,0.520,6,1,99,0.01,0
1,Regra_alta_carga_baixo_ganho,11967,10196,85.201,30,0.251,0.294,3,0,99,0.00,0
2,Apenas_alta_carga,11967,10196,85.201,328,2.741,3.217,6,0,99,0.00,0
